# 05 · Batch pipeline

Fan books across processes with `runner` (recode_pdf / OCR aren't thread-safe,
so each book runs in its own process). Runs are **resumable** — a finished
book's distribution PDF is the sentinel — and append a JSONL report the `stats`
notebook and the live `monitor` read out of band.

In [ ]:
import sys
from pathlib import Path

# project importable when the kernel isn't the project .venv
sys.path.insert(0, str(Path.cwd().parent))

from evilflowers_books_digitalizer.runtime import load_runtime
from evilflowers_books_digitalizer.cache import LocalCache

# ONE source of truth for cache_dir + output_dir (paths resolve to the project
# root regardless of the notebook's working directory) — this is the unified
# cache every notebook and the production runner share.
rt = load_runtime()
cache = LocalCache(rt.cache_dir)
print("cache :", rt.cache_dir)
print("output:", rt.output_dir)

In [ ]:
from evilflowers_books_digitalizer.runner import run_source

# small, sequential run on the laptop (max_parallel=1 is notebook-safe).
src = rt.source_keys[0]
report = run_source(src, max_parallel=1, limit=2)
report

For real batches use the CLI (logs + the live TUI monitor):

```bash
evilflowers-digitalizer run-source svf --config configs/pipeline.toml
evilflowers-digitalizer monitor          # live per-faculty progress
evilflowers-digitalizer run-corpus       # every faculty, resumable
```